# 02 · 加入触点位置，我们多知道了什么？

**事件顺序 + 三维坐标 → 传播方向 → 空间信息检验 → 投影与场**

先运行 Notebook 01。本篇接着使用它新算出的事件，完成 Fig2。


In [ ]:
import _setup
import numpy as np
from analysis import from_events as analysis
from analysis.src.topic5_interictal_direction_rose import fit_event_directions_3d
from plotting import computational as plot
from workflows.tutorial import load_events, TutorialOutput, save_fields

events, xyz, config = load_events()
out = TutorialOutput('fig2')


## Fig2A · 先把通道放回真实空间

每一行坐标与一个通道名称对应。接下来用这些位置解释 Notebook 01 中的先后顺序。


In [ ]:
out.show(plot.contact_geometry(xyz), 'fig2_A_geometry')


## Fig2B · 顺序对应哪个方向？

对每个事件拟合 $rank = a + \beta\cdot xyz$。rank 越大越晚，因此 $\beta/\|\beta\|$ 指向 early → late。按前一本的时序模板分组观察方向分布。


In [ ]:
clustered = analysis.cluster_events(events)
ranks = np.where(events['participating'], events['ranks'], np.nan)
directions = fit_event_directions_3d(ranks, xyz)
out.show(plot.rank_directions(directions, clustered), 'fig2_B_rank_gradient')


## Fig2B · 空间信息是否改善了预测？

用不同记录块训练和测试 Timing 与 Timing+Space。测试事件只按 rank 分配模板，再用方向评分：右图的置换基线用于判断增益能否由偶然对应解释。


In [ ]:
directions, gain = analysis.direction_and_crossfit(events, xyz, n_null=1000, n_train_null=50)
out.show(plot.crossfit_scores(gain), 'fig2_B_crossfit')


## Fig2C · 怎样从触点得到一张场图？

完成检验后，用全部事件拟合描述性模板。以 TA 的传播轴定义同一个投影平面，再用参与率加权的 Gaussian 核平滑 earliness（$-z(rank)$）。这样两模板可在同一坐标系中比较；弱支持区留白。


In [ ]:
clustered = analysis.spatial_templates(events, xyz, directions)
field_record = analysis.template_fields(events, clustered, xyz, n_boot=200,
    subject_id=config['subject'], subject=config['subject'], dataset=config.get('dataset', 'raw_demo'))
field_a = analysis.illustrative_grid(events, clustered, xyz, 0)
field_b = analysis.illustrative_grid(events, clustered, xyz, 1, plane=field_a['plane'])
out.show(plot.field_panels([field_a, field_b],
    field_record['axis_pair'].get('strict_stability_pass', False)), 'fig2_C_fields')


## Fig2D · 两个模板的空间场有多相似？

在共同平面上计算场的相关，打乱 TB 的触点身份并重新平滑，得到比较基线。这衡量场的对应程度，区别于两条方向轴的夹角。


In [ ]:
similarity = analysis.shared_field_similarity(field_record, n_perm=1000)
out.show(plot.shared_field_null(similarity), 'fig2_D_shared_field_null')


## 从数据描述走向模型

以上是 E1146 短例的空间描述，场图并未直接测量连续组织的传播速度。[Notebook 03](03_snn_model_fit_fig4_fig5.ipynb) 接着问：怎样的网络可以产生类似顺序？投影公式、稳定性检验及临床能量场接口见 [方法说明](../METHODS_AND_PANELS.md#notebook2fig2及临床能量场接口)。


In [ ]:
save_fields(out, gain, clustered, field_record, field_a, field_b, similarity)
